# 06 — Baja California Manual Scoring

Manually runs the transform and load steps for Baja California using
the `.txt` files already extracted in `data/raw/Baja_California/`.

All scoring logic is imported directly from `tasks/transform.py` and
`tasks/load.py` to stay in sync with the Airflow pipeline.

In [ ]:
%pip install -q \
    langchain \
    langchain-anthropic \
    langchain-community \
    langchain-text-splitters \
    langchain-core \
    langgraph \
    sentence-transformers \
    faiss-cpu \
    pdfplumber \
    pymongo \
    pyyaml \
    python-dotenv

## 1. Setup & Imports

In [ ]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add project root to sys.path so tasks.* imports work from the notebook
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=False)

# Import scoring primitives directly from the pipeline task
from tasks.transform import (
    MARKERS,
    ComplianceRecord,
    GraphState,
    _build_graph,
    _build_vector_store,
)
from tasks.load import _load_mongo_config, _build_mongo_uri

from pymongo import MongoClient, UpdateOne

print("Imports OK")

## 2. Load Extracted Texts

Read all `.txt` files produced by notebook 05 from
`data/raw/Baja_California/`.

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "Baja_California"

txt_files = sorted(RAW_DIR.glob("*.txt"))
texts = []
for p in txt_files:
    content = p.read_text(encoding="utf-8").strip()
    if content:
        texts.append(content)

total_chars = sum(len(t) for t in texts)
print(f"Files found    : {len(txt_files)}")
print(f"Non-empty files: {len(texts)}")
print(f"Total chars    : {total_chars:,}")

if not texts:
    raise RuntimeError(
        f"No .txt files found in {RAW_DIR}. "
        "Run notebook 05 first to extract the PDFs."
    )

## 3. Build Vector Store

Uses the same `CharacterTextSplitter` (chunk_size=1000, chunk_overlap=200)
and `paraphrase-multilingual-MiniLM-L12-v2` embeddings as `tasks/transform.py`.
This step downloads the model on first run (~120 MB, cached locally after that).

In [ ]:
retriever = _build_vector_store(texts)
print("Vector store built and retriever ready.")

## 4. Run Devil's Advocate Scoring

Runs the LangGraph `retrieve → challenge → rebut → verdict` graph for each
of the 6 compliance markers (L1, L2, L3, C1, C2, C3).
Each marker makes 3 Claude API calls. Expect ~2–4 minutes total.

In [ ]:
STATE_NAME = "Baja_California"

graph   = _build_graph()
records = []

for marker_key, marker_meta in MARKERS.items():
    initial_state: GraphState = {
        "state_name":    STATE_NAME,
        "marker_key":    marker_key,
        "marker_meta":   marker_meta,
        "retriever":     retriever,
        "chunks":        [],
        "challenges":    [],
        "rebuttals":     [],
        "score":         0,
        "justification": "",
        "cited_article": "N/A",
    }

    try:
        final_state = graph.invoke(initial_state)
        record = ComplianceRecord(
            state         = STATE_NAME,
            marker        = marker_key,
            score         = final_state["score"],
            justification = final_state["justification"],
            cited_article = final_state["cited_article"],
        )
        records.append(record.model_dump())
        print(f"[{marker_key}] score={record.score}  cited={record.cited_article}")
    except Exception as exc:
        print(f"[{marker_key}] ERROR: {exc}")
        records.append(ComplianceRecord(
            state         = STATE_NAME,
            marker        = marker_key,
            score         = 0,
            justification = f"Pipeline error: {exc}",
            cited_article = "N/A",
        ).model_dump())

print(f"\nScoring complete: {len(records)} records")

## 5. Save to JSON

Saves the 6 compliance records to `data/output/scores_Baja_California.json`.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUTPUT_DIR / "scores_Baja_California.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} records to {out_path}")

## 6. Upsert to MongoDB

Reads connection details from `config/mongo.yaml` and upserts all 6 records
into the Atlas collection using `(state, marker)` as the composite key.
Re-running this cell overwrites previous scores for the same state/marker pair.

In [ ]:
cfg        = _load_mongo_config()
uri        = _build_mongo_uri(cfg)
client     = MongoClient(uri, serverSelectionTimeoutMS=10_000)
collection = client[cfg["database"]][cfg["collection"]]

operations = [
    UpdateOne(
        filter={"state": r["state"], "marker": r["marker"]},
        update={"$set": r},
        upsert=True,
    )
    for r in records
]

result  = collection.bulk_write(operations, ordered=False)
touched = result.upserted_count + result.modified_count
client.close()

print(f"MongoDB upsert complete: {touched} records touched")
print(f"  upserted={result.upserted_count}  modified={result.modified_count}")